# TESS Data Pickle Reader and Processor

## Necessary Libraries

In [20]:
import os
import io
from io import BytesIO
import torch
from torch.utils.data import Dataset
import numpy as np
from numpy import save
from sklearn.model_selection import train_test_split
import pandas as pd

## Load TESS PKL Data
#### Choose either LOCAL or AWS code cell to pull in TESS pkl data.

In [2]:
LOCAL

set up local file path location

localpath = '../tess_data/'


iterate through file location to collect pke file names

file_names = []
for file in os.listdir(localpath):
    if file.endswith('global.pkl'):
        file_names.append(file)

In [3]:
# # S3

# # implement boto3 connection
# import boto3

# # set up required locations
# filepath = './tess_data'
# S3_BUCKET = 'preprocess-tess-data-bucket'
# S3_PREFIX = 'tess_data/'

# # code for accessing s3 bucket and pulling pkl file names
# s3_client = boto3.client('s3')
# paginator = s3_client.get_paginator('list_objects_v2')
# response_iterator = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX, Delimiter = '/')

# file_names = []
# for response in response_iterator:
#     for object_data in response['Contents']:
#         key = object_data['Key']
        
#         if key.endswith('global.pkl'):
#             file_names.append(key)

## Process Tess Data
#### Choose Local or AWS

In [23]:
# LOCAL VERSION

# reference: https://gitlab.com/frontierdevelopmentlab/exoplanets/exonet-pytorch/-/blob/master/exonet.py
# class that loads data from pkl files
class TessDataLoader(Dataset):
    
    '''
    
    PURPOSE: DATA LOADER FOR KEPLER LIGHT CURVES
    INPUT: PATH TO DIRECTOR WITH LIGHT CURVES + INFO FILES
    OUTPUT: LOCAL + GLOBAL VIEWS, LABELS
    
    '''

    def __init__(self):
        ### Initialize files
        ### list of global, local, and info files (assumes certain names of files)
        self.flist_global = self.__load_file_list__('global.pkl')
        self.flist_local = self.__load_file_list__('local.pkl')
        self.flist_info = self.__load_file_list__('info.pkl')
        
        print(len(self.flist_global))
        
        ### list of whitened centroid files
        self.flist_global_cen = self.__load_file_list__('global_cen.pkl')
        self.flist_local_cen = self.__load_file_list__('local_cen.pkl')
        
        ### ids = {TIC}_{TCE}
        self.ids = np.sort([(x.split('/')[-1]).split('_')[1] + '_' + (x.split('/')[-1]).split('_')[2] for x in self.flist_global])

    def __len__(self):

        return self.ids.shape[0]

    def __getitem__(self, idx):


        ### grab local and global views
        data_global = self.__load_contents__(self.flist_global[idx])
        data_local = self.__load_contents__(self.flist_local[idx])

        ### grab centroid views
        data_global_cen = self.__load_contents__(self.flist_global_cen[idx])
        data_local_cen = self.__load_contents__(self.flist_local_cen[idx])
        
        ### info file contains: [0]kic, [1]tce, [2]period, [3]epoch, [4]duration, [5]label)
        data_info = self.__load_contents__(self.flist_info[idx])

        return (data_local, data_global, data_local_cen, data_global_cen, data_info[3:]), data_info[2]

    def __load_file_list__(self, filename_endswith):
        file_names = []
        for file in os.listdir(localpath):
            if file.endswith(filename_endswith):
                file_names.append(os.path.join(localpath, file))
        return np.sort(file_names)

    def __load_contents__(self, filename):
        contents = np.load(filename, allow_pickle = True)
        return contents

In [ ]:
# # S3 VERSION

# # reference: https://gitlab.com/frontierdevelopmentlab/exoplanets/exonet-pytorch/-/blob/master/exonet.py
# # class that loads data from pkl files
# class TessDataLoader(Dataset):
    
#     '''
    
#     PURPOSE: DATA LOADER FOR KEPLER LIGHT CURVES
#     INPUT: PATH TO DIRECTOR WITH LIGHT CURVES + INFO FILES
#     OUTPUT: LOCAL + GLOBAL VIEWS, LABELS
    
#     '''

#     def __init__(self):
#         ### Initialize S3 bucket
#         self.s3_client = boto3.client('s3')
#         ### list of global, local, and info files (assumes certain names of files)
#         self.flist_global = self.__load_file_list__('global.pkl')
#         self.flist_local = self.__load_file_list__('local.pkl')
#         self.flist_info = self.__load_file_list__('info.pkl')
        
#         print(len(self.flist_global))
        
#         ### list of whitened centroid files
#         self.flist_global_cen = self.__load_file_list__('global_cen.pkl')
#         self.flist_local_cen = self.__load_file_list__('local_cen.pkl')
        
#         ### ids = {TIC}_{TCE}
#         self.ids = np.sort([(x.split('/')[-1]).split('_')[1] + '_' + (x.split('/')[-1]).split('_')[2] for x in self.flist_global])

#     def __len__(self):

#         return self.ids.shape[0]

#     def __getitem__(self, idx):


#         ### grab local and global views
#         data_global = self.__load_contents__(self.flist_global[idx])
#         data_local = self.__load_contents__(self.flist_local[idx])

#         ### grab centroid views
#         data_global_cen = self.__load_contents__(self.flist_global_cen[idx])
#         data_local_cen = self.__load_contents__(self.flist_local_cen[idx])
        
#         ### info file contains: [0]kic, [1]tce, [2]period, [3]epoch, [4]duration, [5]label)
#         data_info = self.__load_contents__(self.flist_info[idx])

#         return (data_local, data_global, data_local_cen, data_global_cen, data_info[3:]), data_info[2]

#     def __load_file_list__(self, filename_endswith):
#         paginator = self.s3_client.get_paginator('list_objects_v2')
#         response_iterator = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX, Delimiter='/')

#         file_names = []
#         for response in response_iterator:
#             for object_data in response['Contents']:
#                 key = object_data['Key']
#                 if key.endswith(filename_endswith):
#                     file_names.append(key)

#         return np.sort(file_names)

#     def __load_contents__(self, filename):
#         bytes_data = io.BytesIO()
#         self.s3_client.download_fileobj(S3_BUCKET, filename, bytes_data)
#         bytes_data.seek(0)
#         contents = np.load(bytes_data, allow_pickle=True)

#         return contents

In [5]:
tess_data = TessDataLoader()

3545


In [6]:
# look through some of the data to verify type and shape
count = 0

for data in tess_data:
    print("len(data):", len(data))
    print("len(data[0]):", len(data[0]))
    print("shape data[0][0]: (data_local):", data[0][0].shape)
    print("shape data[0][1]: (data_global):", data[0][1].shape)
    print("shape data[0][2]: (data_local_cen):", data[0][2].shape)
    print("shape data[0][3]: (data_global_cen):", data[0][3].shape)
    print("shape data[0][4]: (data_info[6:]):", data[0][4].shape)
    print("data[1]:", data[1])
    print()

    count += 1
    if count == 2:
        break

len(data): 2
len(data[0]): 5
shape data[0][0]: (data_local): (101,)
shape data[0][1]: (data_global): (1001,)
shape data[0][2]: (data_local_cen): (101,)
shape data[0][3]: (data_global_cen): (1001,)
shape data[0][4]: (data_info[6:]): (17,)
data[1]: 1.0

len(data): 2
len(data[0]): 5
shape data[0][0]: (data_local): (101,)
shape data[0][1]: (data_global): (1001,)
shape data[0][2]: (data_local_cen): (101,)
shape data[0][3]: (data_global_cen): (1001,)
shape data[0][4]: (data_info[6:]): (17,)
data[1]: 0.0



## Convert to Numpy -  Split Data Into Train/Test/Val - Save Files

In [7]:
# unload all of the data from pytorch dataset to numpy and save
x = []
y = []

curr = 0
total = len(tess_data)
print("Loading validation data:")
for x_data, y_data in tess_data:
    print(curr, "/", total)
    if not isinstance(y_data, np.float64):
        print(f'ERROR: {y_data}')
    curr += 1
    
    x.append(x_data)
    y.append(y_data)

Loading validation data:
0 / 3545
1 / 3545
2 / 3545
3 / 3545
4 / 3545
5 / 3545
6 / 3545
7 / 3545
8 / 3545
9 / 3545
10 / 3545
11 / 3545
12 / 3545
13 / 3545
14 / 3545
15 / 3545
16 / 3545
17 / 3545
18 / 3545
19 / 3545
20 / 3545
21 / 3545
22 / 3545
23 / 3545
24 / 3545
25 / 3545
26 / 3545
27 / 3545
28 / 3545
29 / 3545
30 / 3545
31 / 3545
32 / 3545
33 / 3545
34 / 3545
35 / 3545
36 / 3545
37 / 3545
38 / 3545
39 / 3545
40 / 3545
41 / 3545
42 / 3545
43 / 3545
44 / 3545
45 / 3545
46 / 3545
47 / 3545
48 / 3545
49 / 3545
50 / 3545
51 / 3545
52 / 3545
53 / 3545
54 / 3545
55 / 3545
56 / 3545
57 / 3545
58 / 3545
59 / 3545
60 / 3545
61 / 3545
62 / 3545
63 / 3545
64 / 3545
65 / 3545
66 / 3545
67 / 3545
68 / 3545
69 / 3545
70 / 3545
71 / 3545
72 / 3545
73 / 3545
74 / 3545
75 / 3545
76 / 3545
77 / 3545
78 / 3545
79 / 3545
80 / 3545
81 / 3545
82 / 3545
83 / 3545
84 / 3545
85 / 3545
86 / 3545
87 / 3545
88 / 3545
89 / 3545
90 / 3545
91 / 3545
92 / 3545
93 / 3545
94 / 3545
95 / 3545
96 / 3545
97 / 3545
98 / 

In [8]:
def train_test_split_dataset(data, target, train_ratio, validation_ratio, test_ratio):
    x_train, x_test, y_train, y_test = train_test_split(data, target, test_size=1-train_ratio, shuffle=True, stratify=target)
    
    x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=test_ratio/(test_ratio + validation_ratio), shuffle=True, stratify=y_test)
    
    return x_train, y_train, x_test, y_test, x_val, y_val

In [9]:
# 80% tran, 10% test, 10% validation
x_train, y_train, x_test, y_test, x_val, y_val = train_test_split_dataset(x, y, 0.8, 0.1, 0.1)

pd.Series(y_val).value_counts()

In [17]:
# Converts X and y train/test/val sets from lists into NumPy arrays
x_val = np.asarray(x_val, dtype=object)
y_val = np.asarray(y_val, dtype=np.float32)
x_test = np.asarray(x_test, dtype=object)
y_test = np.asarray(y_test, dtype=np.float32)
x_train = np.asarray(x_train, dtype=object)
y_train = np.asarray(y_train, dtype=np.float32)

print(f'x_val shape: {x_val.shape}')
print(f'x_val[0][0] shape: {x_val[0][0].shape}')
print(f'x_val[0][1] shape: {x_val[0][1].shape}')
print(f'x_val[0][2] shape: {x_val[0][2].shape}')
print(f'x_val[0][3] shape: {x_val[0][3].shape}')
print(f'x_val[0][4] shape: {x_val[0][4].shape}')
print(f'y_val shape: {y_val.shape}')

x_val shape: (354, 5)
x_val[0][0] shape: (101,)
x_val[0][1] shape: (1001,)
x_val[0][2] shape: (101,)
x_val[0][3] shape: (1001,)
x_val[0][4] shape: (17,)
y_val shape: (354,)


## Save the NumPy Arrays
#### Choose Local or AWS

In [21]:
#LOCAL

save("../outputs/val_x_data.npy", x_val)
save("../outputs/val_y_data.npy", y_val)
save("../outputs/test_x_data.npy", x_test)
save("../outputs/test_y_data.npy", y_test)
save("../outputs/train_x_data.npy", x_train)
save("../outputs/train_y_data.npy", y_train)

In [ ]:
# #S3

# def save_to_s3(array, s3_file_path):
    
#     '''
#     array = the numpy array (x_val, y_train, etc)
#     s3_file_path = where to save files within S3 bucket
#     Save array to buffer and uploads to S3 bucket specified location
#     '''
    
#     npy_buffer = BytesIO()
#     np.save(npy_buffer, array)  # Save array to the buffer
#     npy_buffer.seek(0)  # Rewind buffer to the beginning
#     s3_client.upload_fileobj(npy_buffer, S3_BUCKET, s3_file_path)

# # Save the validation, test, and training data to S3
# save_to_s3(x_val, 'outputs/val_x_data.npy')
# save_to_s3(y_val, 'outputs/val_y_data.npy')
# save_to_s3(x_test, 'outputs/test_x_data.npy')
# save_to_s3(y_test, 'outputs/test_y_data.npy')
# save_to_s3(x_train, 'outputs/train_x_data.npy')
# save_to_s3(y_train, 'outputs/train_y_data.npy')

# print("Data saved to S3 successfully.")